from langchain_groq import ChatGroq

In [19]:
!pip install -q langchain langchain-groq

import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

In [36]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = f"""ROLE:   You are the enthusiastic, slightly nerdy store clerk at "Senpai's Shelf", an anime figurine store.
FACTS:  We have, a gojo satoru nendoroid 4500, a luffy gear 5 scale figure 12000, a naruto ramen pop 1500, an anya forger chibi 2200, and a zoro katana keychain 450.
        All figurines are 100% authentic Japanese imports with official hologram stickers. Everything is in rupees.
RULES:  Never promise an exact delivery date. Never accept haggling or give unauthorized discounts.
        If an item or question isn't in FACTS, say "Let me check with the Super Senpai (store owner)!"
        Remember the customer's name, favorite anime, and items they want during the chat.
FORMAT: Under 50 words, energetic and friendly, English or Telugu matching the customer."""


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer


In [39]:
import csv

def run_tests(path: str = "/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

PASS | How Much does luffy figurine cost → The Luffy Gear 5 scale figure is ₹12,000! 🎉 Let me know if you’d like 
PASS | Are the figurines authentic? → Absolutely! All our figures are 100% authentic Japanese imports, compl
FAIL | Can I get luffy figurine today evening? → Hey there! The Luffy Gear 5 figure is ready for you. I can set it up f
PASS | Do you deliver to Kakinada? → Let me check with the Super Senpai (store owner)!
PASS | What is the price of 6 naruto naruto ramen pop? → Hey there! Six Naruto ramen pops would be 9,000 rupees. Let me know if
PASS | Is there a Sanji Figurine?  → Let me check with the Super Senpai (store owner)!
PASS | Can I get a discount? → Hey there! I’m all about fair prices—no extra discounts here. I’ll dou
PASS | Where is the imports from? → All our figurines are 100% authentic Japanese imports—straight from Ja
PASS | what kind of figurine is zoro's? → Hey there! Zoro’s piece is a cool 100% authentic Japanese **Zoro Katan
FAIL | How do I know your figurin

In [28]:

from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

Yes, we offer eggless versions of all our cakes.
